# API Endpoint Testing

Notebook em duas partes:

1. **Requisições simples** — explore cada endpoint manualmente
2. **Testes automatizados** — mesma cobertura de `tests/api_tests/` + casos de borda da API

## Endpoints

| Método | Path | Auth | Descrição |
|--------|------|------|-----------|
| GET | `/health` | Não | Liveness |
| GET | `/recommendations/{user_id}` | Sim | Top-N recomendações (score do modelo) |
| GET | `/recommendation/{user_id}` | Sim | Alias do endpoint acima |
| POST | `/recommendations_filtered` | Sim | Recomendações com filtros avançados |
| POST | `/recommendation_filtered` | Sim | Alias do endpoint acima |
| GET | `/metrics` | Sim | Prometheus (default), `?format=datadog` ou `?format=both` |

Auth: header `x-api-key` (API Gateway, stage `v1`).

Configure manualmente (opcional):

```bash
export RECOMMENDATIONS_API_BASE_URL="https://<api-id>.execute-api.us-east-1.amazonaws.com/v1"
export RECOMMENDATIONS_API_KEY="<sua-api-key>"
export RECOMMENDATIONS_TEST_USER_ID="u_0231"
export RECOMMENDATIONS_TEST_COLD_START_USER_ID="u_9999"
```

Ou deixe o notebook resolver URL/key via `terraform output` / SSM.


In [1]:
import json
import os
import subprocess
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import httpx

PROJECT_ROOT = (Path("..") if Path("..").joinpath("terraform").is_dir() else Path(".")).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TERRAFORM_DIR = PROJECT_ROOT / "terraform"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_STAGE = "v1"
KNOWN_USER_ID = os.getenv("RECOMMENDATIONS_TEST_USER_ID", "u_0231")
COLD_START_USER_ID = os.getenv("RECOMMENDATIONS_TEST_COLD_START_USER_ID", "u_9999")

EXPECTED_PROMETHEUS_METRICS = {
    "recommendations_api_requests_total",
    "recommendations_api_errors_total",
    "recommendations_api_cold_start_total",
    "recommendations_api_latency_avg_ms",
}

EXPECTED_DATADOG_METRICS = {
    "recommendations_api.requests.total",
    "recommendations_api.errors.total",
    "recommendations_api.cold_start.total",
    "recommendations_api.latency.avg_ms",
    "recommendations_api.latency.p50_ms",
    "recommendations_api.latency.p95_ms",
}


def _terraform_output(name: str) -> str | None:
    try:
        return subprocess.check_output(
            ["terraform", f"-chdir={TERRAFORM_DIR}", "output", "-raw", name],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def normalize_api_base_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith(f"/{API_STAGE}"):
        return normalized
    if "execute-api" in normalized:
        return f"{normalized}/{API_STAGE}"
    return normalized


def load_api_config() -> tuple[str, str]:
    base_url = os.getenv("RECOMMENDATIONS_API_BASE_URL")
    api_key = os.getenv("RECOMMENDATIONS_API_KEY")

    if not base_url:
        base_url = _terraform_output("recommendations_api_gateway_endpoint")

    if not api_key:
        api_key = _terraform_output("recommendations_api_key")

    if not api_key:
        param_name = _terraform_output("recommendations_api_key_ssm_parameter")
        if param_name:
            import boto3

            ssm = boto3.client("ssm", region_name=AWS_REGION)
            api_key = ssm.get_parameter(Name=param_name, WithDecryption=True)[
                "Parameter"
            ]["Value"]

    if not base_url or not api_key:
        raise RuntimeError(
            "Defina RECOMMENDATIONS_API_BASE_URL e RECOMMENDATIONS_API_KEY "
            "ou aplique o Terraform e configure credenciais AWS."
        )

    return normalize_api_base_url(base_url), api_key


def api_headers(*, with_key: bool = True) -> dict[str, str]:
    headers = {"Accept": "application/json"}
    if with_key:
        headers["x-api-key"] = API_KEY
    return headers


def call_api(
    method: str,
    path: str,
    *,
    json_body: dict | None = None,
    with_key: bool = True,
    timeout: float = 30.0,
) -> httpx.Response:
    url = f"{API_BASE_URL}{path}"
    with httpx.Client(timeout=timeout) as client:
        return client.request(
            method,
            url,
            headers=api_headers(with_key=with_key),
            json=json_body,
        )


def show_response(response: httpx.Response, *, label: str = "") -> httpx.Response:
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}{response.request.method} {response.request.url}")
    print(f"{prefix}HTTP {response.status_code}")
    content_type = response.headers.get("content-type", "")
    if "json" in content_type:
        print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    else:
        print(response.text)
    return response


def wait_for_api_health(
    *,
    timeout_seconds: float = 300,
    poll_interval_seconds: float = 10,
) -> None:
    deadline = time.monotonic() + timeout_seconds
    last_status: int | str = "unknown"
    while time.monotonic() < deadline:
        try:
            response = call_api("GET", "/health", with_key=False, timeout=10.0)
            last_status = response.status_code
            if response.status_code == 200:
                return
        except httpx.HTTPError:
            last_status = "connection_error"
        time.sleep(poll_interval_seconds)
    raise TimeoutError(
        f"/health did not return 200 within {timeout_seconds}s (last_status={last_status})"
    )


def warn_if_predictions_table_empty() -> None:
    try:
        from tests.helpers.aws_integration import (
            dynamodb_table_has_items,
            load_terraform_outputs,
        )

        outputs = load_terraform_outputs()
        table_name = outputs.get("predictions_dynamodb_table_name")
        if table_name and not dynamodb_table_has_items(table_name):
            print(
                f"AVISO: tabela DynamoDB '{table_name}' vazia. "
                "Rode model_predict antes dos testes de usuário conhecido."
            )
    except Exception as error:  # noqa: BLE001
        print(f"AVISO: não foi possível verificar DynamoDB ({error}).")


API_BASE_URL, API_KEY = load_api_config()
wait_for_api_health()
warn_if_predictions_table_empty()

print(f"API base URL: {API_BASE_URL}")
print(f"Known user:   {KNOWN_USER_ID}")
print(f"Cold start:   {COLD_START_USER_ID}")


AVISO: não foi possível verificar DynamoDB (No module named 'pytest').
API base URL: https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1
Known user:   u_0231
Cold start:   u_9999


## Requisições simples

Execute cada célula abaixo para inspecionar um endpoint. Ajuste `KNOWN_USER_ID` / `COLD_START_USER_ID` na célula de setup se quiser.


### `GET /health` (público, sem API key)

In [2]:
show_response(call_api("GET", "/health", with_key=False), label="health")

[health] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/health
[health] HTTP 200
{
  "status": "ok"
}


<Response [200 OK]>

### `GET /recommendation/{user_id}`

In [3]:
show_response(
    call_api("GET", f"/recommendation/{KNOWN_USER_ID}"),
    label="recommendation",
)

[recommendation] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/recommendation/u_0231
[recommendation] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 10,
  "recommendations": [
    {
      "product_id": "p_032",
      "score": 0.206472161039946
    },
    {
      "product_id": "p_026",
      "score": 0.2034872902495403
    },
    {
      "product_id": "p_057",
      "score": 0.13124761043915797
    },
    {
      "product_id": "p_059",
      "score": 0.09371419024752797
    },
    {
      "product_id": "p_039",
      "score": 0.09328103329528584
    },
    {
      "product_id": "p_058",
      "score": 0.0903509648974111
    },
    {
      "product_id": "p_000",
      "score": 0.08241557144476351
    },
    {
      "product_id": "p_015",
      "score": 0.08206454678397163
    },
    {
      "product_id": "p_042",
      "score": 0.08138293906541968
    },
    {
      "product_id": "p_022",
      "score": 0.07568379533880892
    }
  ]
}


<Response [200 OK]>

### `GET /recommendations/{user_id}` (alias + cold start)

In [4]:
print("--- usuário existente ---")
show_response(
    call_api("GET", f"/recommendations/{KNOWN_USER_ID}"),
    label="recommendations",
)

print("\n--- cold start ---")
show_response(
    call_api("GET", f"/recommendations/{COLD_START_USER_ID}"),
    label="cold_start",
)

--- usuário existente ---
[recommendations] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/recommendations/u_0231
[recommendations] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 10,
  "recommendations": [
    {
      "product_id": "p_032",
      "score": 0.206472161039946
    },
    {
      "product_id": "p_026",
      "score": 0.2034872902495403
    },
    {
      "product_id": "p_057",
      "score": 0.13124761043915797
    },
    {
      "product_id": "p_059",
      "score": 0.09371419024752797
    },
    {
      "product_id": "p_039",
      "score": 0.09328103329528584
    },
    {
      "product_id": "p_058",
      "score": 0.0903509648974111
    },
    {
      "product_id": "p_000",
      "score": 0.08241557144476351
    },
    {
      "product_id": "p_015",
      "score": 0.08206454678397163
    },
    {
      "product_id": "p_042",
      "score": 0.08138293906541968
    },
    {
      "product_id": "p_022",
      "score": 0.075683795338

<Response [200 OK]>

### `POST /recommendations_filtered`

In [5]:
baseline = call_api("GET", f"/recommendations/{KNOWN_USER_ID}").json()
excluded_product_id = baseline["recommendations"][0]["product_id"]

show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_product_ids": [excluded_product_id],
            "category": "esporte",
        },
    ),
    label="recommendations_filtered",
)

[recommendations_filtered] POST https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[recommendations_filtered] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 5,
  "category": "esporte",
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 2,
      "price": 353.48,
      "avg_rating": 4.5,
      "popularity_score": 0.326,
      "user_affinity_match": 1,
      "recommendation_score": 0.13124761043915797,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_059",
      "is_cold_start": false,
      "inte

<Response [200 OK]>

### `POST /recommendation_filtered` (alias)

In [6]:
show_response(
    call_api(
        "POST",
        "/recommendation_filtered",
        json_body={"user_id": KNOWN_USER_ID, "limit": 3},
    ),
    label="recommendation_filtered",
)

[recommendation_filtered] POST https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/recommendation_filtered
[recommendation_filtered] HTTP 200
{
  "user_id": "u_0231",
  "cold_start_flag": false,
  "count": 3,
  "category": null,
  "recommendations": [
    {
      "user_id": "u_0231",
      "product_id": "p_032",
      "is_cold_start": false,
      "interactions": 3,
      "price": 24.31,
      "avg_rating": 3.2,
      "popularity_score": 0.125,
      "user_affinity_match": 1,
      "recommendation_score": 0.206472161039946,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_026",
      "is_cold_start": false,
      "interactions": 3,
      "price": 129.92,
      "avg_rating": 4.6,
      "popularity_score": 0.292,
      "user_affinity_match": 1,
      "recommendation_score": 0.2034872902495403,
      "category": "esporte"
    },
    {
      "user_id": "u_0231",
      "product_id": "p_057",
      "is_cold_start": false,
      "interactions": 

<Response [200 OK]>

### Casos de erro (manual)

In [7]:
print("--- user_id inválido (400) ---")
show_response(
    call_api("GET", "/recommendations/invalid_user"),
    label="invalid_user",
)

print("\n--- rota protegida sem API key (401/403) ---")
show_response(
    call_api("GET", f"/recommendations/{KNOWN_USER_ID}", with_key=False),
    label="no_api_key",
)

print("\n--- categoria inválida no filtro (400) ---")
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={"user_id": KNOWN_USER_ID, "category": "invalida"},
    ),
    label="invalid_category",
)

--- user_id inválido (400) ---
[invalid_user] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/recommendations/invalid_user
[invalid_user] HTTP 400
{
  "detail": "user_id must match the pattern u_XXXX (example: u_0231)"
}

--- rota protegida sem API key (401/403) ---
[no_api_key] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/recommendations/u_0231
[no_api_key] HTTP 403
{
  "message": "Forbidden"
}

--- categoria inválida no filtro (400) ---
[invalid_category] POST https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/recommendations_filtered
[invalid_category] HTTP 400
{
  "detail": "unsupported category 'invalida'. allowed=['beleza', 'casa', 'eletronicos', 'esporte', 'livros', 'moda']"
}


<Response [400 Bad Request]>

### `GET /metrics`

In [8]:
show_response(call_api("GET", "/metrics"), label="metrics_prometheus")

[metrics_prometheus] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/metrics
[metrics_prometheus] HTTP 200
# HELP recommendations_api_requests_total Total HTTP requests handled.
# TYPE recommendations_api_requests_total counter
recommendations_api_requests_total 15.0
# HELP recommendations_api_errors_total Total HTTP errors.
# TYPE recommendations_api_errors_total counter
recommendations_api_errors_total 3.0
# HELP recommendations_api_cold_start_total Total cold-start fallbacks.
# TYPE recommendations_api_cold_start_total counter
recommendations_api_cold_start_total 2.0
# HELP recommendations_api_latency_ms Request latency in milliseconds.
# TYPE recommendations_api_latency_ms summary
recommendations_api_latency_ms_count 15.0
recommendations_api_latency_ms_sum 773.3515229993486
recommendations_api_latency_ms{quantile="0.5"} 17.269842000018798
recommendations_api_latency_ms{quantile="0.95"} 191.70088230015898
# HELP recommendations_api_latency_avg_ms Average request latenc

<Response [200 OK]>

In [9]:
show_response(call_api("GET", "/metrics?format=datadog"), label="metrics_datadog")

[metrics_datadog] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/metrics?format=datadog
[metrics_datadog] HTTP 200
{
  "series": [
    {
      "metric": "recommendations_api.requests.total",
      "type": 1,
      "points": [
        {
          "timestamp": 1785433983,
          "value": 15.0
        }
      ],
      "tags": [
        "service:recommendations_api"
      ]
    },
    {
      "metric": "recommendations_api.errors.total",
      "type": 1,
      "points": [
        {
          "timestamp": 1785433983,
          "value": 3.0
        }
      ],
      "tags": [
        "service:recommendations_api"
      ]
    },
    {
      "metric": "recommendations_api.cold_start.total",
      "type": 1,
      "points": [
        {
          "timestamp": 1785433983,
          "value": 2.0
        }
      ],
      "tags": [
        "service:recommendations_api"
      ]
    },
    {
      "metric": "recommendations_api.latency.count",
      "type": 3,
      "points": [
      

<Response [200 OK]>

In [10]:
show_response(call_api("GET", "/metrics?format=both"), label="metrics_both")

[metrics_both] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/metrics?format=both
[metrics_both] HTTP 200
{
  "prometheus": "# HELP recommendations_api_requests_total Total HTTP requests handled.\n# TYPE recommendations_api_requests_total counter\nrecommendations_api_requests_total 15.0\n# HELP recommendations_api_errors_total Total HTTP errors.\n# TYPE recommendations_api_errors_total counter\nrecommendations_api_errors_total 3.0\n# HELP recommendations_api_cold_start_total Total cold-start fallbacks.\n# TYPE recommendations_api_cold_start_total counter\nrecommendations_api_cold_start_total 2.0\n# HELP recommendations_api_latency_ms Request latency in milliseconds.\n# TYPE recommendations_api_latency_ms summary\nrecommendations_api_latency_ms_count 15.0\nrecommendations_api_latency_ms_sum 773.3515229993486\nrecommendations_api_latency_ms{quantile=\"0.5\"} 17.269842000018798\nrecommendations_api_latency_ms{quantile=\"0.95\"} 191.70088230015898\n# HELP recommendations_api

<Response [200 OK]>

In [11]:
show_response(call_api("GET", "/metrics?format=statsd"), label="metrics_invalid_format")

[metrics_invalid_format] GET https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1/metrics?format=statsd
[metrics_invalid_format] HTTP 400
{
  "detail": "format must be one of: prometheus, datadog, both"
}


<Response [400 Bad Request]>

## Testes automatizados

Espelha `tests/api_tests/test_smoke.py` e `tests/api_tests/test_metrics.py`, com validações extras de contrato (ranking, exclusão de produtos, formatos de métricas).

Execute **depois** das requisições simples (ou rode só esta seção — os testes fazem as chamadas necessárias).


In [12]:
@dataclass
class CheckResult:
    name: str
    passed: bool
    detail: str = ""


@dataclass
class ApiTestReport:
    checks: list[CheckResult] = field(default_factory=list)

    def add(self, name: str, passed: bool, detail: str = "") -> None:
        self.checks.append(CheckResult(name=name, passed=passed, detail=detail))

    def assert_all_passed(self) -> None:
        failures = [check for check in self.checks if not check.passed]
        if failures:
            lines = "\n".join(f"- {item.name}: {item.detail}" for item in failures)
            raise AssertionError(f"Checks failed:\n{lines}")

    def print_results(self) -> None:
        for check in self.checks:
            status = "PASS" if check.passed else "FAIL"
            print(f"[{status}] {check.name}: {check.detail}")


def parse_prometheus_metrics(text: str) -> dict[str, float]:
    metrics: dict[str, float] = {}
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "{" in line:
            value = float(line.rsplit(" ", 1)[1])
            if 'quantile="0.5"' in line:
                metrics["recommendations_api_latency_ms_p50"] = value
            elif 'quantile="0.95"' in line:
                metrics["recommendations_api_latency_ms_p95"] = value
            continue
        name, value = line.rsplit(" ", 1)
        metrics[name] = float(value)
    return metrics


def validate_readme_metrics(metrics_text: str, *, min_requests: int) -> ApiTestReport:
    report = ApiTestReport()
    parsed = parse_prometheus_metrics(metrics_text)

    missing = EXPECTED_PROMETHEUS_METRICS - set(parsed)
    report.add(
        "prometheus_required_metrics",
        not missing,
        f"missing={sorted(missing)}" if missing else "all required counters/gauges present",
    )

    requests_total = parsed.get("recommendations_api_requests_total", 0.0)
    errors_total = parsed.get("recommendations_api_errors_total", 0.0)
    cold_start_total = parsed.get("recommendations_api_cold_start_total", 0.0)
    p50 = parsed.get("recommendations_api_latency_ms_p50", -1.0)
    p95 = parsed.get("recommendations_api_latency_ms_p95", -1.0)

    report.add(
        "request_count_available",
        requests_total >= min_requests,
        f"requests_total={requests_total}, expected>={min_requests}",
    )
    report.add(
        "error_rate_available",
        "recommendations_api_errors_total" in parsed,
        f"errors_total={errors_total}",
    )
    report.add(
        "cold_start_counter_available",
        cold_start_total >= 1,
        f"cold_start_total={cold_start_total}",
    )
    report.add("latency_p50_available", p50 >= 0, f"p50_ms={p50}")
    report.add("latency_p95_available", p95 >= 0, f"p95_ms={p95}")
    report.add("latency_p95_gte_p50", p95 >= p50, f"p50_ms={p50}, p95_ms={p95}")

    error_rate = (errors_total / requests_total) if requests_total else 0.0
    report.add(
        "error_rate_reasonable",
        error_rate <= 0.5,
        f"error_rate={error_rate:.2%}",
    )
    return report


def validate_datadog_metrics(payload: dict[str, Any]) -> ApiTestReport:
    report = ApiTestReport()
    series = payload.get("series", [])
    report.add("datadog_has_series", bool(series), f"series_count={len(series)}")

    metrics = {entry["metric"] for entry in series}
    missing = EXPECTED_DATADOG_METRICS - metrics
    report.add(
        "datadog_required_metrics",
        not missing,
        f"missing={sorted(missing)}" if missing else "all required metrics present",
    )

    for entry in series:
        metric_name = entry.get("metric", "<unknown>")
        report.add(
            f"datadog_type_valid::{metric_name}",
            entry.get("type") in {1, 3},
            f"type={entry.get('type')}",
        )
        points = entry.get("points", [])
        report.add(
            f"datadog_points_valid::{metric_name}",
            bool(points) and "timestamp" in points[0] and "value" in points[0],
            f"points={len(points)}",
        )
    return report


### Smoke tests (`test_smoke.py`)

In [13]:
report = ApiTestReport()

report.add(
    "rest_api_stage_in_base_url",
    API_BASE_URL.endswith(f"/{API_STAGE}"),
    API_BASE_URL,
)

health = call_api("GET", "/health", with_key=False)
report.add("health_status_200", health.status_code == 200, f"status={health.status_code}")
report.add(
    "health_payload",
    health.json().get("status") == "ok",
    str(health.json()),
)
report.add(
    "health_is_public",
    call_api("GET", "/health", with_key=False).status_code == 200,
    "health accessible without x-api-key",
)

recommendation = call_api("GET", f"/recommendations/{KNOWN_USER_ID}")
rec_body = recommendation.json()
report.add(
    "recommendations_status_200",
    recommendation.status_code == 200,
    f"status={recommendation.status_code}",
)
report.add(
    "recommendations_has_user_id",
    rec_body.get("user_id") == KNOWN_USER_ID,
    f"user_id={rec_body.get('user_id')}",
)
report.add(
    "recommendations_has_ranked_items",
    rec_body.get("count", 0) > 0 and bool(rec_body.get("recommendations")),
    f"count={rec_body.get('count')}",
)
report.add(
    "recommendations_has_score",
    all("score" in item for item in rec_body.get("recommendations", [])),
    "each item includes score",
)
scores = [item["score"] for item in rec_body.get("recommendations", [])]
report.add(
    "recommendations_ranked_by_score",
    scores == sorted(scores, reverse=True),
    f"scores={scores[:3]}...",
)
report.add(
    "recommendations_not_cold_start",
    rec_body.get("cold_start_flag") is False,
    f"cold_start_flag={rec_body.get('cold_start_flag')}",
)

singular = call_api("GET", f"/recommendation/{KNOWN_USER_ID}")
report.add(
    "recommendation_alias_status_200",
    singular.status_code == 200,
    f"status={singular.status_code}",
)
report.add(
    "recommendation_alias_same_user",
    singular.json().get("user_id") == KNOWN_USER_ID,
    str(singular.json().get("user_id")),
)

cold_start = call_api("GET", f"/recommendations/{COLD_START_USER_ID}")
cold_body = cold_start.json()
report.add(
    "cold_start_status_200",
    cold_start.status_code == 200,
    f"status={cold_start.status_code}",
)
report.add(
    "cold_start_flag_true",
    cold_body.get("cold_start_flag") is True,
    f"cold_start_flag={cold_body.get('cold_start_flag')}",
)
report.add(
    "cold_start_has_recommendations",
    cold_body.get("count", 0) > 0,
    f"count={cold_body.get('count')}",
)

excluded_product_id = rec_body["recommendations"][0]["product_id"]
filtered = call_api(
    "POST",
    "/recommendations_filtered",
    json_body={
        "user_id": KNOWN_USER_ID,
        "limit": 5,
        "exclude_product_ids": [excluded_product_id],
        "category": "esporte",
    },
)
filtered_body = filtered.json()
report.add(
    "filtered_status_200",
    filtered.status_code == 200,
    f"status={filtered.status_code}",
)
report.add(
    "filtered_respects_limit",
    filtered_body.get("count", 0) <= 5,
    f"count={filtered_body.get('count')}",
)
report.add(
    "filtered_has_detailed_fields",
    all(
        {"product_id", "recommendation_score", "category"} <= set(item)
        for item in filtered_body.get("recommendations", [])
    ),
    "product_id, recommendation_score and category present",
)
report.add(
    "filtered_excludes_product",
    all(
        item.get("product_id") != excluded_product_id
        for item in filtered_body.get("recommendations", [])
    ),
    f"excluded={excluded_product_id}",
)
report.add(
    "filtered_category_filter",
    filtered_body.get("category") == "esporte"
    and all(
        item.get("category") == "esporte"
        for item in filtered_body.get("recommendations", [])
    ),
    f"category={filtered_body.get('category')}",
)

filtered_alias = call_api(
    "POST",
    "/recommendation_filtered",
    json_body={"user_id": KNOWN_USER_ID, "limit": 3},
)
report.add(
    "recommendation_filtered_alias_status_200",
    filtered_alias.status_code == 200,
    f"status={filtered_alias.status_code}",
)
report.add(
    "recommendation_filtered_alias_respects_limit",
    filtered_alias.json().get("count", 0) <= 3,
    f"count={filtered_alias.json().get('count')}",
)

invalid = call_api("GET", "/recommendations/invalid_user")
report.add(
    "invalid_user_returns_400",
    invalid.status_code == 400,
    f"status={invalid.status_code}",
)

invalid_singular = call_api("GET", "/recommendation/bad_user")
report.add(
    "invalid_user_singular_returns_400",
    invalid_singular.status_code == 400,
    f"status={invalid_singular.status_code}",
)

invalid_filtered = call_api(
    "POST",
    "/recommendations_filtered",
    json_body={"user_id": KNOWN_USER_ID, "category": "invalida"},
)
report.add(
    "filtered_invalid_category_returns_400",
    invalid_filtered.status_code == 400,
    f"status={invalid_filtered.status_code}",
)

unauthorized = call_api("GET", f"/recommendations/{KNOWN_USER_ID}", with_key=False)
report.add(
    "protected_route_requires_api_key",
    unauthorized.status_code in {401, 403},
    f"status={unauthorized.status_code}",
)

report.print_results()
report.assert_all_passed()
print("Endpoint smoke tests passed.")


[PASS] rest_api_stage_in_base_url: https://ha7w0htdd3.execute-api.us-east-1.amazonaws.com/v1
[PASS] health_status_200: status=200
[PASS] health_payload: {'status': 'ok'}
[PASS] health_is_public: health accessible without x-api-key
[PASS] recommendations_status_200: status=200
[PASS] recommendations_has_user_id: user_id=u_0231
[PASS] recommendations_has_ranked_items: count=10
[PASS] recommendations_has_score: each item includes score
[PASS] recommendations_ranked_by_score: scores=[0.206472161039946, 0.2034872902495403, 0.13124761043915797]...
[PASS] recommendations_not_cold_start: cold_start_flag=False
[PASS] recommendation_alias_status_200: status=200
[PASS] recommendation_alias_same_user: u_0231
[PASS] cold_start_status_200: status=200
[PASS] cold_start_flag_true: cold_start_flag=True
[PASS] cold_start_has_recommendations: count=10
[PASS] filtered_status_200: status=200
[PASS] filtered_respects_limit: count=5
[PASS] filtered_has_detailed_fields: product_id, recommendation_score and ca

### Métricas (`test_metrics.py` + formatos)

In [14]:
metrics_report = ApiTestReport()

prometheus_response = call_api("GET", "/metrics")
metrics_text = prometheus_response.text
metrics_report.add(
    "metrics_status_200",
    prometheus_response.status_code == 200,
    f"status={prometheus_response.status_code}",
)
metrics_report.add(
    "metrics_prometheus_content_type",
    "text/plain" in (prometheus_response.headers.get("content-type") or ""),
    prometheus_response.headers.get("content-type", ""),
)

readme_checks = validate_readme_metrics(metrics_text, min_requests=6)
metrics_report.checks.extend(readme_checks.checks)

datadog_response = call_api("GET", "/metrics?format=datadog")
metrics_report.add(
    "metrics_datadog_status_200",
    datadog_response.status_code == 200,
    f"status={datadog_response.status_code}",
)
metrics_report.add(
    "metrics_datadog_content_type",
    "application/json" in (datadog_response.headers.get("content-type") or ""),
    datadog_response.headers.get("content-type", ""),
)
datadog_checks = validate_datadog_metrics(datadog_response.json())
metrics_report.checks.extend(datadog_checks.checks)

both_response = call_api("GET", "/metrics?format=both")
both_body = both_response.json() if both_response.headers.get("content-type", "").startswith("application/json") else {}
metrics_report.add(
    "metrics_both_status_200",
    both_response.status_code == 200,
    f"status={both_response.status_code}",
)
metrics_report.add(
    "metrics_both_has_prometheus",
    "prometheus" in both_body and "recommendations_api_requests_total" in both_body.get("prometheus", ""),
    "combined payload includes prometheus text",
)
metrics_report.add(
    "metrics_both_has_datadog",
    "datadog" in both_body and bool(both_body.get("datadog", {}).get("series")),
    "combined payload includes datadog series",
)

invalid_format = call_api("GET", "/metrics?format=statsd")
metrics_report.add(
    "metrics_invalid_format_returns_400",
    invalid_format.status_code == 400,
    f"status={invalid_format.status_code}",
)

metrics_report.print_results()
metrics_report.assert_all_passed()
print("Metrics validation passed.")


[PASS] metrics_status_200: status=200
[PASS] metrics_prometheus_content_type: text/plain; version=0.0.4; charset=utf-8
[PASS] prometheus_required_metrics: all required counters/gauges present
[PASS] request_count_available: requests_total=23.0, expected>=6
[PASS] error_rate_available: errors_total=6.0
[PASS] cold_start_counter_available: cold_start_total=3.0
[PASS] latency_p50_available: p50_ms=17.166057000395085
[PASS] latency_p95_available: p95_ms=45.676987700198865
[PASS] latency_p95_gte_p50: p50_ms=17.166057000395085, p95_ms=45.676987700198865
[PASS] error_rate_reasonable: error_rate=26.09%
[PASS] metrics_datadog_status_200: status=200
[PASS] metrics_datadog_content_type: application/json
[PASS] datadog_has_series: series_count=8
[PASS] datadog_required_metrics: all required metrics present
[PASS] datadog_type_valid::recommendations_api.requests.total: type=1
[PASS] datadog_points_valid::recommendations_api.requests.total: points=1
[PASS] datadog_type_valid::recommendations_api.err

### Resumo

In [15]:
passed = sum(1 for check in [*report.checks, *metrics_report.checks] if check.passed)
failed = sum(1 for check in [*report.checks, *metrics_report.checks] if not check.passed)

print(f"Total checks: {passed + failed}")
print(f"Passed:       {passed}")
print(f"Failed:       {failed}")
if failed == 0:
    print("\nTodos os testes da API passaram.")
else:
    raise AssertionError(f"{failed} check(s) failed")


Total checks: 60
Passed:       60
Failed:       0

Todos os testes da API passaram.
